In [3]:
!pip install transformers datasets accelerate gdown

  Using cached transformers-5.13.0-py3-none-any.whl.metadata (32 kB)
  Using cached datasets-5.0.0-py3-none-any.whl.metadata (23 kB)
  Using cached accelerate-1.14.0-py3-none-any.whl.metadata (19 kB)
  Using cached gdown-6.1.0-py3-none-any.whl.metadata (8.3 kB)
  Using cached huggingface_hub-1.22.0-py3-none-any.whl.metadata (14 kB)
  Using cached regex-2026.6.28-cp312-cp312-manylinux2014_x86_64.manylinux_2_17_x86_64.manylinux_2_28_x86_64.whl.metadata (40 kB)
  Using cached tokenizers-0.22.2-cp39-abi3-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (7.3 kB)
  Using cached typer-0.26.8-py3-none-any.whl.metadata (15 kB)
  Using cached safetensors-0.8.0-cp310-abi3-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (4.2 kB)
  Using cached click-8.4.2-py3-none-any.whl.metadata (2.6 kB)
  Using cached hf_xet-1.5.1-cp37-abi3-manylinux2014_x86_64.manylinux_2_17_x86_64.whl.metadata (4.9 kB)
  Using cached pyarrow-24.0.0-cp312-cp312-manylinux_2_28_x86_64.whl.metadata (3.0 kB)
  Using

In [4]:
#!gdown 1kJXtUko-70rNP250KxkiW3OTE7bg1RAo

In [5]:
import numpy as np
import torch

from datasets import Dataset
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score, accuracy_score, classification_report
import pandas as pd
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer
)

In [6]:
df = pd.read_csv("final_dataset.csv")

In [7]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.pipeline import FeatureUnion, Pipeline
from sklearn.metrics import classification_report, f1_score

# ---------------------------------------------------------
# 1) Ucitavanje i priprema podataka
# ---------------------------------------------------------
df = pd.read_csv("final_dataset.csv")

# baciti redove bez teksta (nema sta trenirati)
df = df.dropna(subset=["SADRZAJ"]).reset_index(drop=True)

TOPICS = [
    "euroatlantske_integracije",
    "negiranje_genocida",
    "gradjanska_vs_konstitutivni",
    "izborna_reforma",
]

def make_label(row, topic):
    """samo mentioned / not_mentioned"""
    if not row[f"{topic}_mentioned"]:
        return "not_mentioned"
    return "mentioned"

for topic in TOPICS:
    df[f"{topic}_label"] = df.apply(lambda r: make_label(r, topic), axis=1)

In [8]:
df["TEXT_BERT"] = (
    df["naslov"].fillna("") + ". " +
    df["SADRZAJ"].fillna("")
)

In [9]:
from transformers import EarlyStoppingCallback

def train_bert_for_topic(topic, model_name="classla/bcms-bertic"):
    print("=" * 70)
    print(f"TEMA: {topic} | BERT FINE-TUNING (mentioned / not_mentioned)")
    print("=" * 70)

    labels = ["not_mentioned", "mentioned"]
    label2id = {label: i for i, label in enumerate(labels)}
    id2label = {i: label for label, i in label2id.items()}

    temp = df[["TEXT_BERT", f"{topic}_label"]].dropna().copy()
    temp["label"] = temp[f"{topic}_label"].map(label2id)

    train_df, test_df = train_test_split(
        temp,
        test_size=0.2,
        random_state=42,
        stratify=temp["label"]
    )

    train_ds = Dataset.from_pandas(train_df[["TEXT_BERT", "label"]])
    test_ds = Dataset.from_pandas(test_df[["TEXT_BERT", "label"]])

    tokenizer = AutoTokenizer.from_pretrained(model_name)

    def tokenize(batch):
        return tokenizer(
            batch["TEXT_BERT"],
            truncation=True,
            padding="max_length",
            max_length=512
        )

    train_ds = train_ds.map(tokenize, batched=True)
    test_ds = test_ds.map(tokenize, batched=True)

    train_ds = train_ds.remove_columns(["TEXT_BERT", "__index_level_0__"])
    test_ds = test_ds.remove_columns(["TEXT_BERT", "__index_level_0__"])

    train_ds.set_format("torch")
    test_ds.set_format("torch")

    model = AutoModelForSequenceClassification.from_pretrained(
        model_name,
        num_labels=len(labels),
        id2label=id2label,
        label2id=label2id
    )

    def compute_metrics(eval_pred):
        logits, y_true = eval_pred
        y_pred = np.argmax(logits, axis=1)

        return {
            "accuracy": accuracy_score(y_true, y_pred),
            "macro_f1": f1_score(y_true, y_pred, average="macro", zero_division=0),
            "weighted_f1": f1_score(y_true, y_pred, average="weighted", zero_division=0),
        }

    args = TrainingArguments(
        output_dir=f"./bert_results_{topic}",
        eval_strategy="epoch",
        save_strategy="epoch",
        learning_rate=2e-5,
        per_device_train_batch_size=8,
        per_device_eval_batch_size=8,
        num_train_epochs=10,
        weight_decay=0.01,
        load_best_model_at_end=True,
        metric_for_best_model="macro_f1",
        greater_is_better=True,
        logging_steps=50,
        save_total_limit=1,
        report_to="none"
    )

    trainer = Trainer(
        model=model,
        args=args,
        train_dataset=train_ds,
        eval_dataset=test_ds,
        compute_metrics=compute_metrics,
        callbacks=[EarlyStoppingCallback(early_stopping_patience=2)]
    )

    trainer.train()

    pred = trainer.predict(test_ds)
    y_pred = np.argmax(pred.predictions, axis=1)
    y_true = pred.label_ids

    print(classification_report(
        y_true,
        y_pred,
        target_names=labels,
        zero_division=0
    ))

    macro_f1 = f1_score(y_true, y_pred, average="macro", zero_division=0)

    return {
        "trainer": trainer,
        "model": model,
        "tokenizer": tokenizer,
        "macro_f1": macro_f1,
        "label2id": label2id,
        "id2label": id2label,
    }

In [10]:
bert_result_euro = train_bert_for_topic("euroatlantske_integracije")

TEMA: euroatlantske_integracije | BERT FINE-TUNING (mentioned / not_mentioned)


Map:   0%|          | 0/12004 [00:00<?, ? examples/s]

Map:   0%|          | 0/3002 [00:00<?, ? examples/s]

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] ElectraForSequenceClassification LOAD REPORT from: classla/bcms-bertic
Key                                               | Status     | 
--------------------------------------------------+------------+-
discriminator_predictions.dense.bias              | UNEXPECTED | 
discriminator_predictions.dense_prediction.weight | UNEXPECTED | 
discriminator_predictions.dense_prediction.bias   | UNEXPECTED | 
discriminator_predictions.dense.weight            | UNEXPECTED | 
classifier.dense.bias                             | MISSING    | 
classifier.dense.weight                           | MISSING    | 
classifier.out_proj.weight                        | MISSING    | 
classifier.out_proj.bias                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

               precision    recall  f1-score   support

not_mentioned       0.93      0.94      0.93      2174
    mentioned       0.83      0.81      0.82       828

     accuracy                           0.90      3002
    macro avg       0.88      0.87      0.88      3002
 weighted avg       0.90      0.90      0.90      3002



In [11]:
bert_result_gradanska = train_bert_for_topic("gradjanska_vs_konstitutivni")

TEMA: gradjanska_vs_konstitutivni | BERT FINE-TUNING (mentioned / not_mentioned)


Map:   0%|          | 0/12004 [00:00<?, ? examples/s]

Map:   0%|          | 0/3002 [00:00<?, ? examples/s]

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] ElectraForSequenceClassification LOAD REPORT from: classla/bcms-bertic
Key                                               | Status     | 
--------------------------------------------------+------------+-
discriminator_predictions.dense.bias              | UNEXPECTED | 
discriminator_predictions.dense_prediction.weight | UNEXPECTED | 
discriminator_predictions.dense_prediction.bias   | UNEXPECTED | 
discriminator_predictions.dense.weight            | UNEXPECTED | 
classifier.dense.bias                             | MISSING    | 
classifier.dense.weight                           | MISSING    | 
classifier.out_proj.weight                        | MISSING    | 
classifier.out_proj.bias                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

               precision    recall  f1-score   support

not_mentioned       0.94      0.94      0.94      2373
    mentioned       0.77      0.76      0.77       629

     accuracy                           0.90      3002
    macro avg       0.85      0.85      0.85      3002
 weighted avg       0.90      0.90      0.90      3002



In [12]:
bert_result_izborna = train_bert_for_topic("izborna_reforma")

TEMA: izborna_reforma | BERT FINE-TUNING (mentioned / not_mentioned)


Map:   0%|          | 0/12004 [00:00<?, ? examples/s]

Map:   0%|          | 0/3002 [00:00<?, ? examples/s]

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] ElectraForSequenceClassification LOAD REPORT from: classla/bcms-bertic
Key                                               | Status     | 
--------------------------------------------------+------------+-
discriminator_predictions.dense.bias              | UNEXPECTED | 
discriminator_predictions.dense_prediction.weight | UNEXPECTED | 
discriminator_predictions.dense_prediction.bias   | UNEXPECTED | 
discriminator_predictions.dense.weight            | UNEXPECTED | 
classifier.dense.bias                             | MISSING    | 
classifier.dense.weight                           | MISSING    | 
classifier.out_proj.weight                        | MISSING    | 
classifier.out_proj.bias                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

               precision    recall  f1-score   support

not_mentioned       0.97      0.99      0.98      2743
    mentioned       0.87      0.71      0.78       259

     accuracy                           0.97      3002
    macro avg       0.92      0.85      0.88      3002
 weighted avg       0.96      0.97      0.96      3002



In [13]:
bert_result_genocid = train_bert_for_topic("negiranje_genocida")

TEMA: negiranje_genocida | BERT FINE-TUNING (mentioned / not_mentioned)


Map:   0%|          | 0/12004 [00:00<?, ? examples/s]

Map:   0%|          | 0/3002 [00:00<?, ? examples/s]

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] ElectraForSequenceClassification LOAD REPORT from: classla/bcms-bertic
Key                                               | Status     | 
--------------------------------------------------+------------+-
discriminator_predictions.dense.bias              | UNEXPECTED | 
discriminator_predictions.dense_prediction.weight | UNEXPECTED | 
discriminator_predictions.dense_prediction.bias   | UNEXPECTED | 
discriminator_predictions.dense.weight            | UNEXPECTED | 
classifier.dense.bias                             | MISSING    | 
classifier.dense.weight                           | MISSING    | 
classifier.out_proj.weight                        | MISSING    | 
classifier.out_proj.bias                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Epoch,Training Loss,Validation Loss,Accuracy,Macro F1,Weighted F1
1,0.230898,0.152657,0.953364,0.901131,0.953173
2,0.138389,0.182091,0.950700,0.900147,0.951599
3,0.067063,0.233820,0.947702,0.896158,0.949139


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

               precision    recall  f1-score   support

not_mentioned       0.97      0.97      0.97      2588
    mentioned       0.84      0.82      0.83       414

     accuracy                           0.95      3002
    macro avg       0.90      0.90      0.90      3002
 weighted avg       0.95      0.95      0.95      3002



In [14]:
results = {
    "euroatlantske_integracije": bert_result_euro,
    "gradjanska_vs_konstitutivni": bert_result_gradanska,
    "izborna_reforma": bert_result_izborna,
    "negiranje_genocida": bert_result_genocid,
}

for topic, res in results.items():
    save_path = f"./bertic_{topic}_final"
    res["trainer"].save_model(save_path)
    res["tokenizer"].save_pretrained(save_path)
    print(f"Sacuvano: {save_path}")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Sacuvano: ./bertic_euroatlantske_integracije_final


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Sacuvano: ./bertic_gradjanska_vs_konstitutivni_final


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Sacuvano: ./bertic_izborna_reforma_final


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Sacuvano: ./bertic_negiranje_genocida_final


In [15]:
import os, shutil

os.makedirs("bertic_models", exist_ok=True)

topics = [
    "euroatlantske_integracije",
    "gradjanska_vs_konstitutivni",
    "izborna_reforma",
    "negiranje_genocida",
]

for topic in topics:
    src = f"./bertic_{topic}_final"
    dst = f"./bertic_models/{topic}"
    if os.path.exists(src):
        shutil.copytree(src, dst, dirs_exist_ok=True)

shutil.make_archive("bertic_models", "zip", "bertic_models")
print("Gotovo: bertic_models.zip")

Gotovo: bertic_models.zip


# Predict

In [18]:
def predict(text, model, tokenizer, device):
    inputs = tokenizer(
        text,
        truncation=True,
        padding="max_length",
        max_length=512,
        return_tensors="pt"
    ).to(device)

    with torch.no_grad():
        logits = model(**inputs).logits
        probs = torch.softmax(logits, dim=1)
        pred_id = torch.argmax(probs, dim=1).item()

    return model.config.id2label[pred_id], probs[0][pred_id].item()


def predict_bin(text, euro_m, izborna_m, genocid_m, grad_m,
                 euro_tok, izborna_tok, genocid_tok, grad_tok, device):

    euro_p = predict(text, euro_m, euro_tok, device)[0]
    izborna_p = predict(text, izborna_m, izborna_tok, device)[0]
    genocid_p = predict(text, genocid_m, genocid_tok, device)[0]
    grad_p = predict(text, grad_m, grad_tok, device)[0]

    return {
        "izborna_p": izborna_p,
        "euro_p": euro_p,
        "genocid_p": genocid_p,
        "grad_p": grad_p,
    }

In [21]:
device = "cuda" if torch.cuda.is_available() else "cpu"
euro_tok, euro_m = AutoTokenizer.from_pretrained("./bertic_euroatlantske_integracije_final"), \
                   AutoModelForSequenceClassification.from_pretrained("./bertic_euroatlantske_integracije_final").to(device).eval()

izborna_tok, izborna_m = AutoTokenizer.from_pretrained("./bertic_izborna_reforma_final"), \
                         AutoModelForSequenceClassification.from_pretrained("./bertic_izborna_reforma_final").to(device).eval()

genocid_tok, genocid_m = AutoTokenizer.from_pretrained("./bertic_negiranje_genocida_final"), \
                         AutoModelForSequenceClassification.from_pretrained("./bertic_negiranje_genocida_final").to(device).eval()

grad_tok, grad_m = AutoTokenizer.from_pretrained("./bertic_gradjanska_vs_konstitutivni_final"), \
                   AutoModelForSequenceClassification.from_pretrained("./bertic_gradjanska_vs_konstitutivni_final").to(device).eval()




Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

In [24]:
text = """
Građevinski radovi u punom su jeku na periferiji Berbere, obalnog grada u samoproglašenoj Republici Somaliland.
Pored moderne luke ubrzano se proširuje i aerodrom, za koji se tvrdi da bi mogao postati vojna baza namijenjena Ujedinjenim Arapskim Emiratima (UAE), Sjedinjenim Američkim Državama i Izraelu.

Prema pisanju Le Mondea, radovi su uslijedili nakon što je Izrael 26. decembra 2025. godine priznao nezavisnost Somalilanda.

Cilj tog poteza je uspostavljanje izraelskog uporišta u Adenskom zaljevu, nedaleko od jemenske obale, gdje djeluju Huti, koji su u sukobu s Izraelom zbog njihove agresije na Gazu.

Satelitski snimci koje je analizirao Le Monde pokazuju opsežne zemljane radove južno od piste. Iskopani su brojni rovovi koji su, prema evropskom sigurnosnom izvoru, vjerovatno namijenjeni skladištenju municije ili goriva.

Vojni stručnjak kojeg je kontaktirao Le Monde tvrdi da radove izvode UAE u skladu sa sporazumom o vojnoj saradnji sa Somalilandom iz 2017. godine. Zaposlenik aerodroma također je izjavio da Emirati grade objekte za potrebe izraelskih i američkih partnera.

Analitičari Međunarodnog instituta za strateške studije (IISS) navode da nasipi i platforme izgrađeni tokom prošle godine ukazuju na planove za raspoređivanje sistema protivzračne odbrane, sličnih onima koje UAE već koriste na aerodromu Bosaso u Puntlandu.

Iako Izrael i Somaliland zvanično negiraju postojanje odbrambenog sporazuma, Le Monde piše da vojna saradnja već postoji. Obavještajni službenici Somalilanda navodno su prolazili obuku u Tel Avivu, dok su izraelske vojne delegacije posjećivale Hargeisu i Berberu.

Berbera ima veliki strateški značaj zbog položaja na ulazu u Crveno more i piste duže od četiri kilometra, izgrađene još u vrijeme Sovjetskog Saveza. Emirati su od 2017. godine obnovili pistu, izgradili vojne hangare i pomorski terminal za ratne brodove, a pristup aerodromu danas je ograničen.

Prema sigurnosnim izvorima iz istočne Afrike, izraelska vojska već obilazi bazu i mogla bi je koristiti za operacije prema Jemenu. Istovremeno, američka komanda za Afriku (AFRICOM) redovno šalje delegacije u Berberu, dok Washington traži alternativu svojoj bazi u Džibutiju, gdje je prisutna i kineska vojska.
"""

In [25]:
result = predict_bin("Ovaj tekst govori o genocidu i evropskim integracijama...",
                      euro_m, izborna_m, genocid_m, grad_m,
                      euro_tok, izborna_tok, genocid_tok, grad_tok, device)
print(result)

{'izborna_p': 'not_mentioned', 'euro_p': 'mentioned', 'genocid_p': 'mentioned', 'grad_p': 'not_mentioned'}
